<!-- NOTEBOOK_OVERVIEW -->
# 1. RoBERTa + IBVS Late-Fusion Hybrid

## 2. Introduction
This notebook runs one bounded transformer-anchored hybrid experiment. It keeps `roberta-base` as the frozen predictive backbone, computes IBVS v2 structured evidence separately, and trains a shallow logistic decision layer on top of `[RoBERTa probability + IBVS features]` using ID-only data.

## 3. Workflow Steps
1. Load the active processed split (`A`, `B`, or `C`) and validate the expected partitions.
2. Fine-tune a single `roberta-base` classifier on `train`, using `val` as the internal checkpoint selection split.
3. Compute RoBERTa probabilities and IBVS v2 feature vectors for `train`, `val`, `test`, and all OOD sets.
4. Fit two bounded late-fusion variants on ID data only:
   1. `HYBRID_ROBERTA_IBVS_TOTAL_LOGREG`
   2. `HYBRID_ROBERTA_IBVS_STRUCTURED_LOGREG`
5. Select the meta-classifier regularization and deployment threshold on `val` only.
6. Evaluate the frozen hybrids on `test`, `ood_test`, `ood_test_injection`, and `ood_test_injection_standard` using the standard dissertation metrics.
7. Export canonical split metrics and optional secondary OOD/bin diagnostics.

## 4. Evaluation Notes
1. This is a late-fusion decision-layer experiment, not a retrained transformer architecture.
2. No OOD tuning or threshold selection is allowed; all selection uses ID `val` only.
3. Reported metrics include `Accuracy`, `Macro-F1`, `AUC-PR`, `ROC-AUC`, and `TPR@{1%,5%,10% FPR}`.
4. `Macro-F1` reflects thresholded deployment behavior, while `AUC-PR`, `ROC-AUC`, and `TPR@fixed FPR` reflect score quality.


In [1]:
# Cell Purpose: Import experiment dependencies, configure randomness, and expose project modules.
# 1) Imports + runtime config

from pathlib import Path
import gc
import os
import sys

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [2]:
# Cell Purpose: Load the active processed split, validate partition integrity, and import evaluation/model helpers.
# 2) Dataset + helper setup

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluation.eval_metrics import (
    evaluate_predictions,
    results_to_dataframe,
    best_threshold_by_macro_f1,
)
from src.common.notebook_utils import safe_qcut, text_stats
from src.models import (
    TransformerBaselineConfig,
    train_transformer_baseline,
    predict_transformer_probabilities,
    RobertaIbvsLateFusionConfig,
    build_ibvs_feature_frame,
    fit_late_fusion_logreg,
    predict_late_fusion_probabilities,
)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

SPLIT_TAG = os.getenv("SPLIT_TAG", "B").strip().upper()
WRITE_MINIMAL_OUTPUTS = os.getenv("WRITE_MINIMAL_OUTPUTS", "1").strip().lower() not in {"0", "false", "no"}
print("WRITE_MINIMAL_OUTPUTS:", WRITE_MINIMAL_OUTPUTS)

expected_filename_by_split = {
    "A": "jailbreak_benchmarks_processed_v2.csv",
    "B": "jailbreak_benchmarks_processed_v2_splitB.csv",
    "C": "jailbreak_benchmarks_processed_v2_splitC.csv",
}
if SPLIT_TAG not in expected_filename_by_split:
    raise ValueError("SPLIT_TAG must be 'A', 'B', or 'C'.")
processed_filename = expected_filename_by_split[SPLIT_TAG]
processed_path = DATA_PROCESSED / processed_filename
if not processed_path.exists():
    raise FileNotFoundError(processed_path)

df = pd.read_csv(processed_path)
print(f"Loaded {processed_path.name}: {df.shape}")

expected_output_suffix = f"split{SPLIT_TAG}"
print("Expected output suffix:", expected_output_suffix)

required_splits = {"train", "val", "test", "ood_test"}
observed_splits = set(df["split"].astype(str).unique())
missing = required_splits - observed_splits
if missing:
    raise ValueError(f"Missing required split(s): {sorted(missing)}")

df_train = df[df["split"] == "train"].copy()
df_val = df[df["split"] == "val"].copy()
df_test = df[df["split"] == "test"].copy()

OOD_SPLIT_ORDER = ["ood_test", "ood_test_injection", "ood_test_injection_standard"]
df_ood_map = {}
for split_name in OOD_SPLIT_ORDER:
    d = df[df["split"] == split_name].copy()
    if not d.empty:
        df_ood_map[split_name] = d

for name, d in [("train", df_train), ("val", df_val), ("test", df_test)] + list(df_ood_map.items()):
    if d.empty:
        raise ValueError(f"Split '{name}' is empty.")
    print(f"{name:26s} {d.shape} {d['label'].value_counts().to_dict()}")

y_train = df_train["label"].to_numpy(dtype=int)
y_val = df_val["label"].to_numpy(dtype=int)
y_test = df_test["label"].to_numpy(dtype=int)
y_ood_map = {split_name: d["label"].to_numpy(dtype=int) for split_name, d in df_ood_map.items()}

X_train_text = df_train["prompt_text"].astype(str).tolist()
X_val_text = df_val["prompt_text"].astype(str).tolist()
X_test_text = df_test["prompt_text"].astype(str).tolist()
X_ood_text_map = {split_name: d["prompt_text"].astype(str).tolist() for split_name, d in df_ood_map.items()}


WRITE_MINIMAL_OUTPUTS: False
Loaded jailbreak_benchmarks_processed_v2_splitC.csv: (6424, 16)
Expected output suffix: splitC
train                      (694, 16) {1: 504, 0: 190}
val                        (149, 16) {1: 109, 0: 40}
test                       (149, 16) {1: 108, 0: 41}
ood_test                   (768, 16) {0: 384, 1: 384}
ood_test_injection         (678, 16) {0: 339, 1: 339}
ood_test_injection_standard (3986, 16) {1: 1993, 0: 1993}


In [3]:
# Cell Purpose: Train the frozen RoBERTa backbone, compute RoBERTa scores, build IBVS feature tables, and fit bounded late-fusion variants.
# 3) RoBERTa backbone + IBVS late-fusion models

print("Protocol note: this notebook uses ID-only training and validation. No OOD tuning is performed.")

ROBETRA_CONFIG = TransformerBaselineConfig(
    model_name="roberta-base",
    output_label="TRANSFORMER_ROBERTA_BASE",
    max_length=256,
    learning_rate=2e-5,
    num_epochs=3,
    train_batch_size=8,
    eval_batch_size=16,
    weight_decay=0.01,
    warmup_ratio=0.1,
)

roberta_artifacts = train_transformer_baseline(
    X_train_text,
    y_train,
    X_val_text,
    y_val,
    config=ROBETRA_CONFIG,
    seed=RANDOM_SEED,
)

roberta_train_prob = predict_transformer_probabilities(
    roberta_artifacts.model,
    roberta_artifacts.tokenizer,
    X_train_text,
    batch_size=ROBETRA_CONFIG.eval_batch_size,
    max_length=ROBETRA_CONFIG.max_length,
    device=roberta_artifacts.device,
)
roberta_val_prob = roberta_artifacts.val_probabilities
roberta_test_prob = predict_transformer_probabilities(
    roberta_artifacts.model,
    roberta_artifacts.tokenizer,
    X_test_text,
    batch_size=ROBETRA_CONFIG.eval_batch_size,
    max_length=ROBETRA_CONFIG.max_length,
    device=roberta_artifacts.device,
)
roberta_ood_prob_map = {
    split_name: predict_transformer_probabilities(
        roberta_artifacts.model,
        roberta_artifacts.tokenizer,
        texts,
        batch_size=ROBETRA_CONFIG.eval_batch_size,
        max_length=ROBETRA_CONFIG.max_length,
        device=roberta_artifacts.device,
    )
    for split_name, texts in X_ood_text_map.items()
}

ibvs_train_df = build_ibvs_feature_frame(X_train_text)
ibvs_val_df = build_ibvs_feature_frame(X_val_text)
ibvs_test_df = build_ibvs_feature_frame(X_test_text)
ibvs_ood_df_map = {split_name: build_ibvs_feature_frame(texts) for split_name, texts in X_ood_text_map.items()}

META_VARIANTS = [
    ("HYBRID_ROBERTA_IBVS_TOTAL_LOGREG", "total"),
    ("HYBRID_ROBERTA_IBVS_STRUCTURED_LOGREG", "structured"),
]
C_GRID = [0.1, 1.0, 3.0, 10.0]

all_metric_frames = []
secondary_ood_frames = {}
inference_cache = {}

for model_label, feature_mode in META_VARIANTS:
    best_bundle = None
    for C in C_GRID:
        meta_cfg = RobertaIbvsLateFusionConfig(
            output_label=model_label,
            feature_mode=feature_mode,
            C=C,
        )
        meta_artifacts = fit_late_fusion_logreg(
            roberta_train_prob,
            ibvs_train_df,
            y_train,
            config=meta_cfg,
        )
        val_prob = predict_late_fusion_probabilities(meta_artifacts, roberta_val_prob, ibvs_val_df)
        t_star, val_macro = best_threshold_by_macro_f1(y_val, val_prob, n_grid=1001)
        candidate = {
            "artifacts": meta_artifacts,
            "val_prob": val_prob,
            "threshold": float(t_star),
            "val_macro": float(val_macro),
            "C": float(C),
        }
        if best_bundle is None or candidate["val_macro"] > best_bundle["val_macro"]:
            best_bundle = candidate

    threshold = float(best_bundle["threshold"])
    val_macro = float(best_bundle["val_macro"])
    chosen_C = float(best_bundle["C"])
    meta_artifacts = best_bundle["artifacts"]

    note_base = (
        f"policy=macro_f1; t*={threshold:.3f}; val_macro_f1={val_macro:.4f}; "
        f"score=late_fusion_logreg; base=TRANSFORMER_ROBERTA_BASE; "
        f"feature_mode={feature_mode}; C={chosen_C}; "
        f"train_score_source={meta_artifacts.config.train_score_source}; "
        f"backbone={ROBETRA_CONFIG.model_name}; max_len={ROBETRA_CONFIG.max_length}; "
        f"epochs={ROBETRA_CONFIG.num_epochs}; lr={ROBETRA_CONFIG.learning_rate}; "
        f"device={roberta_artifacts.device}"
    )

    split_payloads = [
        ("VAL", y_val, best_bundle["val_prob"], "id"),
        ("TEST", y_test, predict_late_fusion_probabilities(meta_artifacts, roberta_test_prob, ibvs_test_df), "id"),
        (
            "OOD",
            y_ood_map["ood_test"],
            predict_late_fusion_probabilities(
                meta_artifacts,
                roberta_ood_prob_map["ood_test"],
                ibvs_ood_df_map["ood_test"],
            ),
            "ood_test",
        ),
    ]

    result_objs = []
    for split_name, y_true, y_score, ood_name in split_payloads:
        y_pred = (y_score >= threshold).astype(int)
        result_objs.append(
            evaluate_predictions(
                split_name,
                y_true,
                y_pred,
                y_score,
                print_report=False,
                threshold_note=note_base + ("" if ood_name == "id" else f"; ood_name={ood_name}"),
            )
        )

    model_df = results_to_dataframe(model_label, result_objs)
    model_df["eval_track"] = "deployment_threshold"
    model_df["ood_name"] = model_df["split"].map({"OOD": "ood_test"}).fillna("id")
    model_df["base_model"] = "TRANSFORMER_ROBERTA_BASE"
    model_df["meta_model"] = "logistic_regression"
    model_df["hybrid_feature_mode"] = feature_mode
    model_df["meta_C"] = chosen_C
    model_df["stack_train_source"] = meta_artifacts.config.train_score_source
    model_df["backbone_name"] = ROBETRA_CONFIG.model_name
    model_df["max_length"] = ROBETRA_CONFIG.max_length
    model_df["learning_rate"] = ROBETRA_CONFIG.learning_rate
    model_df["num_epochs"] = ROBETRA_CONFIG.num_epochs
    model_df["train_batch_size"] = ROBETRA_CONFIG.train_batch_size
    model_df["eval_batch_size"] = ROBETRA_CONFIG.eval_batch_size
    model_df["weight_decay"] = ROBETRA_CONFIG.weight_decay
    model_df["warmup_ratio"] = ROBETRA_CONFIG.warmup_ratio
    model_df["training_seed"] = RANDOM_SEED
    model_df["device"] = roberta_artifacts.device
    all_metric_frames.append(model_df)

    for ood_name in [name for name in df_ood_map if name != "ood_test"]:
        ood_prob = predict_late_fusion_probabilities(
            meta_artifacts,
            roberta_ood_prob_map[ood_name],
            ibvs_ood_df_map[ood_name],
        )
        ood_pred = (ood_prob >= threshold).astype(int)
        ood_result = evaluate_predictions(
            "OOD",
            y_ood_map[ood_name],
            ood_pred,
            ood_prob,
            print_report=False,
            threshold_note=note_base + f"; ood_name={ood_name}",
        )
        ood_df = results_to_dataframe(model_label, [ood_result])
        ood_df["eval_track"] = "deployment_threshold"
        ood_df["ood_name"] = ood_name
        ood_df["base_model"] = "TRANSFORMER_ROBERTA_BASE"
        ood_df["meta_model"] = "logistic_regression"
        ood_df["hybrid_feature_mode"] = feature_mode
        ood_df["meta_C"] = chosen_C
        ood_df["stack_train_source"] = meta_artifacts.config.train_score_source
        ood_df["backbone_name"] = ROBETRA_CONFIG.model_name
        ood_df["max_length"] = ROBETRA_CONFIG.max_length
        ood_df["learning_rate"] = ROBETRA_CONFIG.learning_rate
        ood_df["num_epochs"] = ROBETRA_CONFIG.num_epochs
        ood_df["train_batch_size"] = ROBETRA_CONFIG.train_batch_size
        ood_df["eval_batch_size"] = ROBETRA_CONFIG.eval_batch_size
        ood_df["weight_decay"] = ROBETRA_CONFIG.weight_decay
        ood_df["warmup_ratio"] = ROBETRA_CONFIG.warmup_ratio
        ood_df["training_seed"] = RANDOM_SEED
        ood_df["device"] = roberta_artifacts.device
        secondary_ood_frames.setdefault(ood_name, []).append(ood_df)

    inference_cache[model_label] = {
        "threshold": threshold,
        "note_base": note_base,
        "feature_mode": feature_mode,
        "meta_C": chosen_C,
        "backbone_name": ROBETRA_CONFIG.model_name,
        "scores": {
            "TEST": predict_late_fusion_probabilities(meta_artifacts, roberta_test_prob, ibvs_test_df),
            **{
                split_name: predict_late_fusion_probabilities(meta_artifacts, roberta_ood_prob_map[split_name], ibvs_ood_df_map[split_name])
                for split_name in df_ood_map
            },
        },
    }

results_df = pd.concat(all_metric_frames, ignore_index=True)
secondary_ood_results = {
    ood_name: pd.concat(frames, ignore_index=True)
    for ood_name, frames in secondary_ood_frames.items()
}
print(results_df[["model", "split", "ood_name", "macro_f1", "tpr_at_1pct_fpr", "tpr_at_5pct_fpr", "tpr_at_10pct_fpr", "hybrid_feature_mode", "meta_C"]].to_string(index=False))

gc.collect()


Protocol note: this notebook uses ID-only training and validation. No OOD tuning is performed.


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


                                model split ood_name  macro_f1  tpr_at_1pct_fpr  tpr_at_5pct_fpr  tpr_at_10pct_fpr hybrid_feature_mode  meta_C
     HYBRID_ROBERTA_IBVS_TOTAL_LOGREG   VAL       id  0.974163         0.944954         0.990826          1.000000               total     0.1
     HYBRID_ROBERTA_IBVS_TOTAL_LOGREG  TEST       id  0.982913         0.564815         1.000000          1.000000               total     0.1
     HYBRID_ROBERTA_IBVS_TOTAL_LOGREG   OOD ood_test  0.783007         0.434896         0.609375          0.684896               total     0.1
HYBRID_ROBERTA_IBVS_STRUCTURED_LOGREG   VAL       id  0.974163         0.944954         0.990826          1.000000          structured     0.1
HYBRID_ROBERTA_IBVS_STRUCTURED_LOGREG  TEST       id  0.982913         0.564815         1.000000          1.000000          structured     0.1
HYBRID_ROBERTA_IBVS_STRUCTURED_LOGREG   OOD ood_test  0.783007         0.434896         0.609375          0.684896          structured     0.1

1833

In [4]:
# Cell Purpose: Persist canonical split-level metrics and optional secondary OOD files for the bounded RoBERTa+IBVS experiment.
# 4) Save metrics outputs

OUT_DIR = PROJECT_ROOT / "experiments" / "results" / "metrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / f"metrics_transformer_hybrid_split{SPLIT_TAG}.csv"
results_df.to_csv(out_path, index=False)
print(f"Saved primary transformer-hybrid metrics: {out_path}")

if not WRITE_MINIMAL_OUTPUTS:
    out_primary_suffix = OUT_DIR / f"metrics_transformer_hybrid_split{SPLIT_TAG}__ood-ood_test.csv"
    results_df.to_csv(out_primary_suffix, index=False)
    print(f"Saved primary OOD-suffixed transformer-hybrid metrics: {out_primary_suffix}")

    for ood_name, df_extra in secondary_ood_results.items():
        out_extra = OUT_DIR / f"metrics_transformer_hybrid_split{SPLIT_TAG}__ood-{ood_name}.csv"
        df_extra.to_csv(out_extra, index=False)
        print(f"Saved secondary transformer-hybrid metrics ({ood_name}): {out_extra}")
else:
    print("WRITE_MINIMAL_OUTPUTS=True: skipped transformer-hybrid OOD-suffixed metrics files.")


Saved primary transformer-hybrid metrics: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_hybrid_splitC.csv
Saved primary OOD-suffixed transformer-hybrid metrics: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_hybrid_splitC__ood-ood_test.csv
Saved secondary transformer-hybrid metrics (ood_test_injection): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_hybrid_splitC__ood-ood_test_injection.csv
Saved secondary transformer-hybrid metrics (ood_test_injection_standard): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_hybrid_splitC__ood-ood_test_injection_standard.csv


In [5]:
# Cell Purpose: Produce bin-level diagnostics for the bounded RoBERTa+IBVS hybrid variants.
# 5) Bin-level diagnostics (length + lexical complexity)

def _build_bin_rows(
    model_label: str,
    prompts_stage: pd.Series,
    y_stage: np.ndarray,
    y_score_stage: np.ndarray,
    *,
    stage_name: str,
    ood_name: str,
):
    rows = []
    meta = inference_cache[model_label]
    threshold = float(meta["threshold"])
    y_pred_stage = (y_score_stage >= threshold).astype(int)

    stats_stage = text_stats(prompts_stage)
    stats_stage["length_bin"] = safe_qcut(stats_stage["token_count"], q=4, prefix="len")
    stats_stage["complexity_bin"] = safe_qcut(stats_stage["lexical_ttr"], q=4, prefix="complex")

    for bin_family in ["length_bin", "complexity_bin"]:
        for bin_label in sorted([b for b in stats_stage[bin_family].dropna().unique()]):
            mask = (stats_stage[bin_family] == bin_label).to_numpy(dtype=bool)
            if mask.sum() == 0:
                continue

            y_s = y_stage[mask]
            p_s = y_pred_stage[mask]
            s_s = y_score_stage[mask]
            result = evaluate_predictions(
                stage_name,
                y_s,
                p_s,
                s_s,
                print_report=False,
                threshold_note=meta["note_base"] + f"; ood_name={ood_name}; bin={bin_family}:{bin_label}",
            )
            row = {
                "model": model_label,
                "eval_track": "deployment_threshold",
                "slice_stage": stage_name,
                "ood_name": ood_name,
                "bin_family": bin_family,
                "bin_label": bin_label,
                "n_total": int(mask.sum()),
                "n_positive": int((y_s == 1).sum()),
                "n_negative": int((y_s == 0).sum()),
                "hybrid_feature_mode": meta["feature_mode"],
                "meta_C": meta["meta_C"],
                "base_model": "TRANSFORMER_ROBERTA_BASE",
                "meta_model": "logistic_regression",
                "backbone_name": meta["backbone_name"],
            }
            row.update(result.__dict__)
            rows.append(row)

    return rows

bin_rows = []
for model_label in sorted(inference_cache):
    bin_rows.extend(
        _build_bin_rows(
            model_label,
            df_test["prompt_text"],
            y_test,
            inference_cache[model_label]["scores"]["TEST"],
            stage_name="TEST",
            ood_name="id",
        )
    )
    for split_name, d in df_ood_map.items():
        bin_rows.extend(
            _build_bin_rows(
                model_label,
                d["prompt_text"],
                y_ood_map[split_name],
                inference_cache[model_label]["scores"][split_name],
                stage_name="OOD",
                ood_name=split_name,
            )
        )

bin_df = pd.DataFrame(bin_rows)

out_bins = OUT_DIR / f"metrics_transformer_hybrid_bins_split{SPLIT_TAG}.csv"
primary_bin_df = bin_df[bin_df["ood_name"].eq("ood_test") | bin_df["ood_name"].eq("id")].copy()
primary_bin_df.to_csv(out_bins, index=False)
print(f"Saved transformer-hybrid bin metrics (primary): {out_bins}")

if not WRITE_MINIMAL_OUTPUTS:
    out_bins_primary_suffix = OUT_DIR / f"metrics_transformer_hybrid_bins_split{SPLIT_TAG}__ood-ood_test.csv"
    primary_bin_df.to_csv(out_bins_primary_suffix, index=False)
    print(f"Saved transformer-hybrid bin metrics primary-suffixed: {out_bins_primary_suffix}")

    for ood_name in sorted([name for name in bin_df["ood_name"].dropna().unique() if name not in {"id", "ood_test"}]):
        extra_bin_df = bin_df[bin_df["ood_name"] == ood_name].copy()
        out_extra = OUT_DIR / f"metrics_transformer_hybrid_bins_split{SPLIT_TAG}__ood-{ood_name}.csv"
        extra_bin_df.to_csv(out_extra, index=False)
        print(f"Saved transformer-hybrid bin metrics ({ood_name}): {out_extra}")
else:
    print("WRITE_MINIMAL_OUTPUTS=True: skipped transformer-hybrid OOD-suffixed bin files.")


Saved transformer-hybrid bin metrics (primary): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_hybrid_bins_splitC.csv
Saved transformer-hybrid bin metrics primary-suffixed: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_hybrid_bins_splitC__ood-ood_test.csv
Saved transformer-hybrid bin metrics (ood_test_injection): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_hybrid_bins_splitC__ood-ood_test_injection.csv
Saved transformer-hybrid bin metrics (ood_test_injection_standard): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_hybrid_bins_splitC__ood-ood_test_injection_standard.csv


<!-- NOTEBOOK_OUTPUT_SUMMARY -->
# 6. Output Summary
1. Writes canonical split-level metrics to `experiments/results/metrics/metrics_transformer_hybrid_split{tag}.csv`.
2. When `WRITE_MINIMAL_OUTPUTS=0`, also writes OOD-suffixed split metrics for:
   1. `ood_test`
   2. `ood_test_injection`
   3. `ood_test_injection_standard`
3. Writes bin-level difficulty diagnostics to `experiments/results/metrics/metrics_transformer_hybrid_bins_split{tag}.csv`.
4. When `WRITE_MINIMAL_OUTPUTS=0`, also writes OOD-suffixed bin diagnostics for each OOD set.
5. These outputs are consumed by the repeatability and canonical aggregation stages after reruns.
